In [1]:
!kaggle datasets list -s test

You must authenticate before you can call the Kaggle API.
Follow the instructions to authenticate at: https://github.com/Kaggle/kaggle-cli/blob/main/docs/README.md#authentication


# Nouvelle section

In [2]:
import subprocess
result = subprocess.run(
    ["find", "/content/drive/MyDrive", "-iname", "*newdataset*"],
    capture_output=True, text=True
)
print(result.stdout)

In [3]:
import subprocess
result = subprocess.run(
    ["find", "/content/drive/MyDrive", "-iname", "*NewDataSet*"],
    capture_output=True, text=True
)
print(result.stdout)

In [4]:
import os
print(os.listdir("/content/drive/MyDrive/MonProjet/NewDataSet"))

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/MonProjet/NewDataSet'

In [ ]:
"""
==========================================================
PIPELINE YOLO26 - CLASSIFICATION (Google Colab + Drive natif)
==========================================================
1) Upload kaggle.json (fenêtre de sélection de fichier)
2) Monte Google Drive (autorisation via lien)
3) Télécharge le dataset depuis Kaggle
4) Analyse + Data cleaning
5) Entraînement + Test avec les meilleurs hyperparamètres
6) Sauvegarde directe dans Drive/MonProjet/YOLO26(2)/...
==========================================================
"""

# ==================================================================
# 0) INSTALLATION
# ==================================================================
!pip install -q ultralytics

# ==================================================================
# 1) (API Kaggle non nécessaire — dataset déjà dans Drive)
# ==================================================================
import os

# ==================================================================
# 2) MONTAGE DE GOOGLE DRIVE
# ==================================================================
from google.colab import drive
drive.mount('/content/drive')

# ==================================================================
# 3) DATASET DÉJÀ DANS DRIVE (pas de téléchargement nécessaire)
# ==================================================================

# ==================================================================
# 4) CONFIGURATION
# ==================================================================
from pathlib import Path
import shutil, hashlib, json, random
from collections import defaultdict
from PIL import Image
import matplotlib.pyplot as plt
import pandas as pd
import torch

# Dataset directement dans Drive, pas besoin de chercher/télécharger
DATASET_PATH = "/content/drive/MyDrive/MonProjet/NewDataSet/concrete-damage-v3"
print(f"[Info] Dataset utilisé : {DATASET_PATH}")
print(f"[Info] Existe ? {Path(DATASET_PATH).exists()}")

WORK_PATH = "/content/newdataset_clean"
IMG_SIZE  = 224
SPLITS    = ["train", "val", "test"]

# Dossier Drive de destination (adapte "MonProjet" si besoin)
DRIVE_BASE = Path("/content/drive/MyDrive/MonProjet/YOLO26(2)")
for d in ["Dataset", "Hyperparametres", "Modeles", "Resultats"]:
    (DRIVE_BASE / d).mkdir(parents=True, exist_ok=True)

# ==================================================================
# 5) ANALYSE EXPLORATOIRE
# ==================================================================
def analyser_dataset(dataset_path):
    print("="*60); print("ANALYSE DU DATASET"); print("="*60)
    stats = []
    for split in SPLITS:
        split_path = Path(dataset_path) / split
        if not split_path.exists():
            print(f"[!] Dossier '{split}' introuvable, ignoré.")
            continue
        for cls_dir in [d for d in split_path.iterdir() if d.is_dir()]:
            images = list(cls_dir.glob("*.*"))
            widths, heights, sizes_ko = [], [], []
            for img_path in images[:200]:
                try:
                    with Image.open(img_path) as img:
                        w, h = img.size
                        widths.append(w); heights.append(h)
                    sizes_ko.append(os.path.getsize(img_path) / 1024)
                except Exception:
                    continue
            stats.append({
                "split": split, "classe": cls_dir.name, "n_images": len(images),
                "largeur_moy": sum(widths)/len(widths) if widths else None,
                "hauteur_moy": sum(heights)/len(heights) if heights else None,
                "poids_moy_ko": sum(sizes_ko)/len(sizes_ko) if sizes_ko else None,
            })
    df = pd.DataFrame(stats)
    print(df.to_string(index=False))
    if not df.empty:
        train_df = df[df["split"] == "train"]
        if not train_df.empty:
            plt.figure(figsize=(8, 4))
            plt.bar(train_df["classe"], train_df["n_images"])
            plt.xticks(rotation=45, ha="right")
            plt.title("Distribution des classes (train)")
            plt.ylabel("Nombre d'images")
            plt.tight_layout()
            plt.savefig("/content/distribution_classes.png")
            plt.close()
            ratio = train_df["n_images"].max() / train_df["n_images"].min()
            print(f"\nRatio classe max/min : {ratio:.2f}")
    return df

# ==================================================================
# 6) DATA CLEANING
# ==================================================================
def detecter_images_corrompues(dataset_path):
    corrompues = []
    for split in SPLITS:
        split_path = Path(dataset_path) / split
        if not split_path.exists():
            continue
        for img_path in split_path.rglob("*.*"):
            try:
                with Image.open(img_path) as img:
                    img.verify()
            except Exception:
                corrompues.append(img_path)
    print(f"Images corrompues détectées : {len(corrompues)}")
    return corrompues

def detecter_doublons(dataset_path):
    hashes = defaultdict(list)
    for split in SPLITS:
        split_path = Path(dataset_path) / split
        if not split_path.exists():
            continue
        for img_path in split_path.rglob("*.*"):
            try:
                with open(img_path, "rb") as f:
                    h = hashlib.md5(f.read()).hexdigest()
                hashes[h].append(img_path)
            except Exception:
                continue
    doublons = []
    for h, paths in hashes.items():
        if len(paths) > 1:
            doublons.extend(paths[1:])
    print(f"Doublons détectés : {len(doublons)}")
    return doublons

def nettoyer_dataset(dataset_path, work_path):
    corrompues = set(detecter_images_corrompues(dataset_path))
    doublons = set(detecter_doublons(dataset_path))
    a_exclure = corrompues | doublons
    if os.path.exists(work_path):
        shutil.rmtree(work_path)
    n_copiees, n_exclues = 0, 0
    for split in SPLITS:
        split_path = Path(dataset_path) / split
        if not split_path.exists():
            continue
        for img_path in split_path.rglob("*.*"):
            if img_path in a_exclure:
                n_exclues += 1
                continue
            rel = img_path.relative_to(dataset_path)
            dest = Path(work_path) / rel
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(img_path, dest)
            n_copiees += 1
    print(f"\nNettoyage terminé : {n_copiees} images copiées, {n_exclues} exclues.")
    return work_path

def creer_val_si_absent(work_path, ratio_val=0.2, seed=42):
    random.seed(seed)
    train_path = Path(work_path) / "train"
    val_path = Path(work_path) / "val"
    if val_path.exists() and any(val_path.iterdir()):
        print("[Split] 'val' déjà présent.")
        return
    if not train_path.exists():
        return
    print(f"[Split] Création automatique de val ({ratio_val:.0%} de train).")
    for cls_dir in train_path.iterdir():
        if not cls_dir.is_dir():
            continue
        images = list(cls_dir.glob("*.*"))
        random.shuffle(images)
        n_val = max(1, int(len(images) * ratio_val))
        dest_dir = val_path / cls_dir.name
        dest_dir.mkdir(parents=True, exist_ok=True)
        for img in images[:n_val]:
            shutil.move(str(img), str(dest_dir / img.name))

# ==================================================================
# 7) HYPERPARAMÈTRES
# ==================================================================
def obtenir_best_params():
    device = 0 if torch.cuda.is_available() else "cpu"
    return dict(
        model="yolo26n-cls.pt", epochs=100, imgsz=IMG_SIZE, batch=64,
        optimizer="auto", lr0=0.001, lrf=0.01, momentum=0.937,
        weight_decay=0.0005, warmup_epochs=3, patience=20, dropout=0.1,
        augment=True, device=device, seed=42, pretrained=True, val=True,
        project="/content/runs/classify", name="YOLO26_2",
    )

# ==================================================================
# 8) ENTRAINEMENT ET TEST
# ==================================================================
def entrainer(work_path, params):
    from ultralytics import YOLO
    model = YOLO(params["model"])
    params = dict(params)
    params["data"] = work_path
    results = model.train(**params)
    return model, results

def tester(model, work_path):
    test_path = Path(work_path) / "test"
    split_a_utiliser = "test" if test_path.exists() else "val"
    metrics = model.val(data=work_path, split=split_a_utiliser)
    print(f"Top-1 accuracy : {metrics.top1:.4f}")
    print(f"Top-5 accuracy : {metrics.top5:.4f}")
    return metrics

# ==================================================================
# 9) SAUVEGARDE DIRECTE DANS DRIVE (déjà monté, simple copie)
# ==================================================================
def sauvegarder_dans_drive(df_stats, params, metrics):
    if df_stats is not None and not df_stats.empty:
        df_stats.to_csv(DRIVE_BASE / "Dataset" / "analyse_dataset.csv", index=False)
    dist_png = Path("/content/distribution_classes.png")
    if dist_png.exists():
        shutil.copy2(dist_png, DRIVE_BASE / "Dataset" / "distribution_classes.png")

    params_propre = {k: (str(v) if not isinstance(v, (int, float, bool, str)) else v)
                      for k, v in params.items()}
    with open(DRIVE_BASE / "Hyperparametres" / "hyperparametres_yolo26.json", "w") as f:
        json.dump(params_propre, f, indent=2, ensure_ascii=False)

    runs_dir = Path("/content/runs")
    for pattern in ["**/weights/best.pt", "**/weights/last.pt"]:
        for p in runs_dir.glob(pattern):
            nom = f"{p.stem}_{p.parent.parent.name}.pt"
            shutil.copy2(p, DRIVE_BASE / "Modeles" / nom)

    for pattern in ["**/results.png", "**/results.csv", "**/confusion_matrix.png",
                     "**/confusion_matrix_normalized.png"]:
        for p in runs_dir.glob(pattern):
            shutil.copy2(p, DRIVE_BASE / "Resultats" / p.name)

    readme = f"""# YOLO26(2) — Classification

## Resultats
- Top-1 accuracy : {metrics.top1:.4f}
- Top-5 accuracy : {metrics.top5:.4f}
"""
    with open(DRIVE_BASE / "README_YOLO26.md", "w") as f:
        f.write(readme)

    print(f"\n✅ Tout est déjà dans ton Drive : {DRIVE_BASE}")


# ==================================================================
# 10) EXECUTION COMPLETE
# ==================================================================
df_stats = analyser_dataset(DATASET_PATH)
clean_path = nettoyer_dataset(DATASET_PATH, WORK_PATH)
creer_val_si_absent(clean_path, ratio_val=0.2)

params = obtenir_best_params()
print(f"\n[Info] Device utilisé : {params['device']}")

model, results = entrainer(clean_path, params)
metrics = tester(model, clean_path)

sauvegarder_dans_drive(df_stats, params, metrics)

In [ ]:
"""
==========================================================
PIPELINE EfficientNetB0 - CLASSIFICATION (Google Colab + Drive)
==========================================================
1) Analyse + Data cleaning (même dataset que YOLO26)
2) Meilleurs hyperparamètres
3) Entraînement + Test avec EfficientNetB0 (torchvision, pré-entraîné ImageNet)
4) Sauvegarde directe dans Drive/MonProjet/EfficientNetB0(2)/...
==========================================================
"""

# ==================================================================
# 0) INSTALLATION (torchvision est déjà présent sur Colab normalement)
# ==================================================================
!pip install -q torch torchvision scikit-learn

import os, json, shutil, random, hashlib, time
from pathlib import Path
from collections import defaultdict

from PIL import Image
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

# ==================================================================
# 1) MONTAGE DRIVE (si pas déjà fait dans cette session)
# ==================================================================
from google.colab import drive
drive.mount('/content/drive')

# ==================================================================
# 2) CONFIGURATION
# ==================================================================
DATASET_PATH = "/content/drive/MyDrive/MonProjet/NewDataSet/concrete-damage-v3"
WORK_PATH    = "/content/newdataset_clean_effnet"
IMG_SIZE     = 224
SPLITS       = ["train", "val", "test"]

DRIVE_BASE = Path("/content/drive/MyDrive/MonProjet/EfficientNetB0(2)")
for d in ["Dataset", "Hyperparametres", "Modeles", "Resultats"]:
    (DRIVE_BASE / d).mkdir(parents=True, exist_ok=True)

print(f"[Info] Dataset source : {DATASET_PATH}")
print(f"[Info] Existe ? {Path(DATASET_PATH).exists()}")

# ==================================================================
# 3) ANALYSE EXPLORATOIRE
# ==================================================================
def analyser_dataset(dataset_path):
    print("="*60); print("ANALYSE DU DATASET"); print("="*60)
    stats = []
    for split in SPLITS:
        split_path = Path(dataset_path) / split
        if not split_path.exists():
            print(f"[!] Dossier '{split}' introuvable, ignoré.")
            continue
        for cls_dir in [d for d in split_path.iterdir() if d.is_dir()]:
            images = list(cls_dir.glob("*.*"))
            widths, heights, sizes_ko = [], [], []
            for img_path in images[:200]:
                try:
                    with Image.open(img_path) as img:
                        w, h = img.size
                        widths.append(w); heights.append(h)
                    sizes_ko.append(os.path.getsize(img_path) / 1024)
                except Exception:
                    continue
            stats.append({
                "split": split, "classe": cls_dir.name, "n_images": len(images),
                "largeur_moy": sum(widths)/len(widths) if widths else None,
                "hauteur_moy": sum(heights)/len(heights) if heights else None,
                "poids_moy_ko": sum(sizes_ko)/len(sizes_ko) if sizes_ko else None,
            })
    df = pd.DataFrame(stats)
    print(df.to_string(index=False))
    if not df.empty:
        train_df = df[df["split"] == "train"]
        if not train_df.empty:
            plt.figure(figsize=(8, 4))
            plt.bar(train_df["classe"], train_df["n_images"])
            plt.xticks(rotation=45, ha="right")
            plt.title("Distribution des classes (train)")
            plt.ylabel("Nombre d'images")
            plt.tight_layout()
            plt.savefig("/content/distribution_classes_effnet.png")
            plt.close()
            ratio = train_df["n_images"].max() / train_df["n_images"].min()
            print(f"\nRatio classe max/min : {ratio:.2f}")
    return df

# ==================================================================
# 4) DATA CLEANING
# ==================================================================
def detecter_images_corrompues(dataset_path):
    corrompues = []
    for split in SPLITS:
        split_path = Path(dataset_path) / split
        if not split_path.exists():
            continue
        for img_path in split_path.rglob("*.*"):
            try:
                with Image.open(img_path) as img:
                    img.verify()
            except Exception:
                corrompues.append(img_path)
    print(f"Images corrompues détectées : {len(corrompues)}")
    return corrompues

def detecter_doublons(dataset_path):
    hashes = defaultdict(list)
    for split in SPLITS:
        split_path = Path(dataset_path) / split
        if not split_path.exists():
            continue
        for img_path in split_path.rglob("*.*"):
            try:
                with open(img_path, "rb") as f:
                    h = hashlib.md5(f.read()).hexdigest()
                hashes[h].append(img_path)
            except Exception:
                continue
    doublons = []
    for h, paths in hashes.items():
        if len(paths) > 1:
            doublons.extend(paths[1:])
    print(f"Doublons détectés : {len(doublons)}")
    return doublons

def nettoyer_dataset(dataset_path, work_path):
    corrompues = set(detecter_images_corrompues(dataset_path))
    doublons = set(detecter_doublons(dataset_path))
    a_exclure = corrompues | doublons
    if os.path.exists(work_path):
        shutil.rmtree(work_path)
    n_copiees, n_exclues = 0, 0
    for split in SPLITS:
        split_path = Path(dataset_path) / split
        if not split_path.exists():
            continue
        for img_path in split_path.rglob("*.*"):
            if img_path in a_exclure:
                n_exclues += 1
                continue
            rel = img_path.relative_to(dataset_path)
            dest = Path(work_path) / rel
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(img_path, dest)
            n_copiees += 1
    print(f"\nNettoyage terminé : {n_copiees} images copiées, {n_exclues} exclues.")
    return work_path

def creer_val_si_absent(work_path, ratio_val=0.2, seed=42):
    random.seed(seed)
    train_path = Path(work_path) / "train"
    val_path = Path(work_path) / "val"
    if val_path.exists() and any(val_path.iterdir()):
        print("[Split] 'val' déjà présent.")
        return
    if not train_path.exists():
        return
    print(f"[Split] Création automatique de val ({ratio_val:.0%} de train).")
    for cls_dir in train_path.iterdir():
        if not cls_dir.is_dir():
            continue
        images = list(cls_dir.glob("*.*"))
        random.shuffle(images)
        n_val = max(1, int(len(images) * ratio_val))
        dest_dir = val_path / cls_dir.name
        dest_dir.mkdir(parents=True, exist_ok=True)
        for img in images[:n_val]:
            shutil.move(str(img), str(dest_dir / img.name))

# ==================================================================
# 5) MEILLEURS HYPERPARAMÈTRES (EfficientNetB0)
# ==================================================================
def obtenir_best_params():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    return dict(
        model="efficientnet_b0 (ImageNet pretrained)",
        imgsz=IMG_SIZE,
        epochs=30,              # EfficientNetB0 converge vite en fine-tuning
        batch=32,                # plus petit que YOLO nano car EffNetB0 est plus lourd (5.3M params)
        optimizer="AdamW",
        lr0=0.0003,               # fine-tuning complet, lr modéré
        weight_decay=1e-4,
        scheduler="CosineAnnealingLR",
        label_smoothing=0.1,
        patience=7,               # early stopping (epochs sans amélioration)
        dropout=0.2,              # dropout par défaut d'EfficientNetB0
        augment=True,
        device=device,
        seed=42,
    )

# ==================================================================
# 6) DATALOADERS
# ==================================================================
def preparer_dataloaders(work_path, params):
    mean = [0.485, 0.456, 0.406]
    std  = [0.229, 0.224, 0.225]

    train_tf = transforms.Compose([
        transforms.RandomResizedCrop(params["imgsz"], scale=(0.8, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])
    eval_tf = transforms.Compose([
        transforms.Resize(int(params["imgsz"] * 1.14)),
        transforms.CenterCrop(params["imgsz"]),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])

    train_ds = datasets.ImageFolder(str(Path(work_path) / "train"), transform=train_tf)
    val_ds   = datasets.ImageFolder(str(Path(work_path) / "val"), transform=eval_tf)

    test_path = Path(work_path) / "test"
    test_ds = datasets.ImageFolder(str(test_path), transform=eval_tf) if test_path.exists() else None

    train_loader = DataLoader(train_ds, batch_size=params["batch"], shuffle=True, num_workers=2)
    val_loader   = DataLoader(val_ds, batch_size=params["batch"], shuffle=False, num_workers=2)
    test_loader  = DataLoader(test_ds, batch_size=params["batch"], shuffle=False, num_workers=2) if test_ds else None

    return train_loader, val_loader, test_loader, train_ds.classes

# ==================================================================
# 7) MODELE
# ==================================================================
def construire_modele(num_classes, device):
    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)
    return model.to(device)

# ==================================================================
# 8) ENTRAINEMENT AVEC EARLY STOPPING
# ==================================================================
def entrainer(model, train_loader, val_loader, params):
    device = params["device"]
    optimizer = torch.optim.AdamW(model.parameters(), lr=params["lr0"], weight_decay=params["weight_decay"])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=params["epochs"])
    criterion = nn.CrossEntropyLoss(label_smoothing=params["label_smoothing"])

    best_val_acc = 0.0
    epochs_sans_amelioration = 0
    historique = {"epoch": [], "train_loss": [], "val_loss": [], "val_acc": []}
    best_model_path = "/content/best_efficientnetb0.pt"

    for epoch in range(1, params["epochs"] + 1):
        t0 = time.time()
        model.train()
        train_loss = 0.0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * imgs.size(0)
        train_loss /= len(train_loader.dataset)

        model.eval()
        val_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                outputs = model(imgs)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * imgs.size(0)
                preds = outputs.argmax(dim=1)
                correct += (preds == labels).sum().item()
                total += labels.size(0)
        val_loss /= len(val_loader.dataset)
        val_acc = correct / total

        scheduler.step()
        dt = time.time() - t0
        print(f"Epoch {epoch}/{params['epochs']} - train_loss={train_loss:.4f} "
              f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} ({dt:.1f}s)")

        historique["epoch"].append(epoch)
        historique["train_loss"].append(train_loss)
        historique["val_loss"].append(val_loss)
        historique["val_acc"].append(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            epochs_sans_amelioration = 0
            torch.save(model.state_dict(), best_model_path)
        else:
            epochs_sans_amelioration += 1
            if epochs_sans_amelioration >= params["patience"]:
                print(f"\n[EarlyStopping] Arrêt à l'epoch {epoch} (meilleur val_acc={best_val_acc:.4f})")
                break

    return best_model_path, historique, best_val_acc

# ==================================================================
# 9) TEST FINAL
# ==================================================================
def tester(model, best_model_path, loader, classes, device):
    model.load_state_dict(torch.load(best_model_path))
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            outputs = model(imgs)
            preds = outputs.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())

    acc = accuracy_score(all_labels, all_preds)
    print(f"\nTest accuracy : {acc:.4f}")
    print(classification_report(all_labels, all_preds, target_names=classes))

    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(6, 5))
    plt.imshow(cm, cmap="Blues")
    plt.title("Matrice de confusion")
    plt.colorbar()
    plt.xticks(range(len(classes)), classes, rotation=45, ha="right")
    plt.yticks(range(len(classes)), classes)
    for i in range(len(classes)):
        for j in range(len(classes)):
            plt.text(j, i, cm[i, j], ha="center", va="center")
    plt.xlabel("Prédiction"); plt.ylabel("Réel"); plt.tight_layout()
    plt.savefig("/content/confusion_matrix_effnet.png")
    plt.close()

    rapport = classification_report(all_labels, all_preds, target_names=classes, output_dict=True)
    return acc, rapport


# ==================================================================
# 10) SAUVEGARDE DANS DRIVE
# ==================================================================
def sauvegarder_dans_drive(df_stats, params, historique, test_acc, rapport, best_model_path):
    if df_stats is not None and not df_stats.empty:
        df_stats.to_csv(DRIVE_BASE / "Dataset" / "analyse_dataset.csv", index=False)
    dist_png = Path("/content/distribution_classes_effnet.png")
    if dist_png.exists():
        shutil.copy2(dist_png, DRIVE_BASE / "Dataset" / "distribution_classes.png")

    params_propre = {k: str(v) for k, v in params.items()}
    with open(DRIVE_BASE / "Hyperparametres" / "hyperparametres_efficientnetb0.json", "w") as f:
        json.dump(params_propre, f, indent=2, ensure_ascii=False)

    shutil.copy2(best_model_path, DRIVE_BASE / "Modeles" / "best_efficientnetb0.pt")

    df_hist = pd.DataFrame(historique)
    df_hist.to_csv(DRIVE_BASE / "Resultats" / "results.csv", index=False)

    plt.figure(figsize=(8, 4))
    plt.plot(df_hist["epoch"], df_hist["train_loss"], label="train_loss")
    plt.plot(df_hist["epoch"], df_hist["val_loss"], label="val_loss")
    plt.legend(); plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title("Courbes de loss")
    plt.tight_layout()
    plt.savefig(DRIVE_BASE / "Resultats" / "results.png")
    plt.close()

    cm_png = Path("/content/confusion_matrix_effnet.png")
    if cm_png.exists():
        shutil.copy2(cm_png, DRIVE_BASE / "Resultats" / "confusion_matrix.png")

    with open(DRIVE_BASE / "Resultats" / "classification_report.json", "w") as f:
        json.dump(rapport, f, indent=2, ensure_ascii=False)

    readme = f"""# EfficientNetB0(2) — Classification

## Resultats
- Test accuracy : {test_acc:.4f}
- Voir Resultats/classification_report.json pour le détail par classe
"""
    with open(DRIVE_BASE / "README_EfficientNetB0.md", "w") as f:
        f.write(readme)

    print(f"\n✅ Tout est dans ton Drive : {DRIVE_BASE}")


# ==================================================================
# 11) EXECUTION COMPLETE
# ==================================================================
df_stats = analyser_dataset(DATASET_PATH)
clean_path = nettoyer_dataset(DATASET_PATH, WORK_PATH)
creer_val_si_absent(clean_path, ratio_val=0.2)

params = obtenir_best_params()
print(f"\n[Info] Device utilisé : {params['device']}")

train_loader, val_loader, test_loader, classes = preparer_dataloaders(clean_path, params)
print(f"[Info] Classes : {classes}")

model = construire_modele(len(classes), params["device"])
best_model_path, historique, best_val_acc = entrainer(model, train_loader, val_loader, params)

loader_test_final = test_loader if test_loader else val_loader
test_acc, rapport = tester(model, best_model_path, loader_test_final, classes, params["device"])

sauvegarder_dans_drive(df_stats, params, historique, test_acc, rapport, best_model_path)

In [ ]:
"""
==========================================================
PIPELINE EfficientNetB0 - CLASSIFICATION (Google Colab + Drive)
==========================================================
1) Analyse + Data cleaning (même dataset que YOLO26)
2) Meilleurs hyperparamètres
3) Entraînement + Test avec EfficientNetB0 (torchvision, pré-entraîné ImageNet)
4) Sauvegarde directe dans Drive/MonProjet/EfficientNetB0(2)/...
==========================================================
"""

# ==================================================================
# 0) INSTALLATION (torchvision est déjà présent sur Colab normalement)
# ==================================================================
!pip install -q torch torchvision scikit-learn

import os, json, shutil, random, hashlib, time
from pathlib import Path
from collections import defaultdict

from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True  # évite le crash sur images tronquées pendant le training
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

# ==================================================================
# 1) MONTAGE DRIVE (si pas déjà fait dans cette session)
# ==================================================================
from google.colab import drive
drive.mount('/content/drive')

# ==================================================================
# 2) CONFIGURATION
# ==================================================================
DATASET_PATH = "/content/drive/MyDrive/MonProjet/NewDataSet/concrete-damage-v3"
WORK_PATH    = "/content/newdataset_clean_effnet"
IMG_SIZE     = 224
SPLITS       = ["train", "val", "test"]

DRIVE_BASE = Path("/content/drive/MyDrive/MonProjet/EfficientNetB0(2)")
for d in ["Dataset", "Hyperparametres", "Modeles", "Resultats"]:
    (DRIVE_BASE / d).mkdir(parents=True, exist_ok=True)

print(f"[Info] Dataset source : {DATASET_PATH}")
print(f"[Info] Existe ? {Path(DATASET_PATH).exists()}")

# ==================================================================
# 3) ANALYSE EXPLORATOIRE
# ==================================================================
def analyser_dataset(dataset_path):
    print("="*60); print("ANALYSE DU DATASET"); print("="*60)
    stats = []
    for split in SPLITS:
        split_path = Path(dataset_path) / split
        if not split_path.exists():
            print(f"[!] Dossier '{split}' introuvable, ignoré.")
            continue
        for cls_dir in [d for d in split_path.iterdir() if d.is_dir()]:
            images = list(cls_dir.glob("*.*"))
            widths, heights, sizes_ko = [], [], []
            for img_path in images[:200]:
                try:
                    with Image.open(img_path) as img:
                        w, h = img.size
                        widths.append(w); heights.append(h)
                    sizes_ko.append(os.path.getsize(img_path) / 1024)
                except Exception:
                    continue
            stats.append({
                "split": split, "classe": cls_dir.name, "n_images": len(images),
                "largeur_moy": sum(widths)/len(widths) if widths else None,
                "hauteur_moy": sum(heights)/len(heights) if heights else None,
                "poids_moy_ko": sum(sizes_ko)/len(sizes_ko) if sizes_ko else None,
            })
    df = pd.DataFrame(stats)
    print(df.to_string(index=False))
    if not df.empty:
        train_df = df[df["split"] == "train"]
        if not train_df.empty:
            plt.figure(figsize=(8, 4))
            plt.bar(train_df["classe"], train_df["n_images"])
            plt.xticks(rotation=45, ha="right")
            plt.title("Distribution des classes (train)")
            plt.ylabel("Nombre d'images")
            plt.tight_layout()
            plt.savefig("/content/distribution_classes_effnet.png")
            plt.close()
            ratio = train_df["n_images"].max() / train_df["n_images"].min()
            print(f"\nRatio classe max/min : {ratio:.2f}")
    return df

# ==================================================================
# 4) DATA CLEANING
# ==================================================================
def detecter_images_corrompues(dataset_path):
    """
    Détecte les images illisibles OU tronquées.
    img.verify() seul ne suffit pas : il vérifie la structure du fichier
    mais peut laisser passer des images tronquées (données pixel incomplètes).
    On force donc un vrai chargement des pixels avec img.load().
    """
    corrompues = []
    for split in SPLITS:
        split_path = Path(dataset_path) / split
        if not split_path.exists():
            continue
        for img_path in split_path.rglob("*.*"):
            try:
                with Image.open(img_path) as img:
                    img.load()  # force le décodage complet des pixels (détecte la troncature)
            except Exception:
                corrompues.append(img_path)
    print(f"Images corrompues/tronquées détectées : {len(corrompues)}")
    return corrompues

def detecter_doublons(dataset_path):
    hashes = defaultdict(list)
    for split in SPLITS:
        split_path = Path(dataset_path) / split
        if not split_path.exists():
            continue
        for img_path in split_path.rglob("*.*"):
            try:
                with open(img_path, "rb") as f:
                    h = hashlib.md5(f.read()).hexdigest()
                hashes[h].append(img_path)
            except Exception:
                continue
    doublons = []
    for h, paths in hashes.items():
        if len(paths) > 1:
            doublons.extend(paths[1:])
    print(f"Doublons détectés : {len(doublons)}")
    return doublons

def nettoyer_dataset(dataset_path, work_path):
    corrompues = set(detecter_images_corrompues(dataset_path))
    doublons = set(detecter_doublons(dataset_path))
    a_exclure = corrompues | doublons
    if os.path.exists(work_path):
        shutil.rmtree(work_path)
    n_copiees, n_exclues = 0, 0
    for split in SPLITS:
        split_path = Path(dataset_path) / split
        if not split_path.exists():
            continue
        for img_path in split_path.rglob("*.*"):
            if img_path in a_exclure:
                n_exclues += 1
                continue
            rel = img_path.relative_to(dataset_path)
            dest = Path(work_path) / rel
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(img_path, dest)
            n_copiees += 1
    print(f"\nNettoyage terminé : {n_copiees} images copiées, {n_exclues} exclues.")
    return work_path

def creer_val_si_absent(work_path, ratio_val=0.2, seed=42):
    random.seed(seed)
    train_path = Path(work_path) / "train"
    val_path = Path(work_path) / "val"
    if val_path.exists() and any(val_path.iterdir()):
        print("[Split] 'val' déjà présent.")
        return
    if not train_path.exists():
        return
    print(f"[Split] Création automatique de val ({ratio_val:.0%} de train).")
    for cls_dir in train_path.iterdir():
        if not cls_dir.is_dir():
            continue
        images = list(cls_dir.glob("*.*"))
        random.shuffle(images)
        n_val = max(1, int(len(images) * ratio_val))
        dest_dir = val_path / cls_dir.name
        dest_dir.mkdir(parents=True, exist_ok=True)
        for img in images[:n_val]:
            shutil.move(str(img), str(dest_dir / img.name))

# ==================================================================
# 5) MEILLEURS HYPERPARAMÈTRES (EfficientNetB0)
# ==================================================================
def obtenir_best_params():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    return dict(
        model="efficientnet_b0 (ImageNet pretrained)",
        imgsz=IMG_SIZE,
        epochs=30,              # EfficientNetB0 converge vite en fine-tuning
        batch=32,                # plus petit que YOLO nano car EffNetB0 est plus lourd (5.3M params)
        optimizer="AdamW",
        lr0=0.0003,               # fine-tuning complet, lr modéré
        weight_decay=1e-4,
        scheduler="CosineAnnealingLR",
        label_smoothing=0.1,
        patience=7,               # early stopping (epochs sans amélioration)
        dropout=0.2,              # dropout par défaut d'EfficientNetB0
        augment=True,
        device=device,
        seed=42,
    )

# ==================================================================
# 6) DATALOADERS
# ==================================================================
def preparer_dataloaders(work_path, params):
    mean = [0.485, 0.456, 0.406]
    std  = [0.229, 0.224, 0.225]

    train_tf = transforms.Compose([
        transforms.RandomResizedCrop(params["imgsz"], scale=(0.8, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])
    eval_tf = transforms.Compose([
        transforms.Resize(int(params["imgsz"] * 1.14)),
        transforms.CenterCrop(params["imgsz"]),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])

    train_ds = datasets.ImageFolder(str(Path(work_path) / "train"), transform=train_tf)
    val_ds   = datasets.ImageFolder(str(Path(work_path) / "val"), transform=eval_tf)

    test_path = Path(work_path) / "test"
    test_ds = datasets.ImageFolder(str(test_path), transform=eval_tf) if test_path.exists() else None

    train_loader = DataLoader(train_ds, batch_size=params["batch"], shuffle=True, num_workers=2)
    val_loader   = DataLoader(val_ds, batch_size=params["batch"], shuffle=False, num_workers=2)
    test_loader  = DataLoader(test_ds, batch_size=params["batch"], shuffle=False, num_workers=2) if test_ds else None

    return train_loader, val_loader, test_loader, train_ds.classes

# ==================================================================
# 7) MODELE
# ==================================================================
def construire_modele(num_classes, device):
    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)
    return model.to(device)

# ==================================================================
# 8) ENTRAINEMENT AVEC EARLY STOPPING
# ==================================================================
def entrainer(model, train_loader, val_loader, params):
    device = params["device"]
    optimizer = torch.optim.AdamW(model.parameters(), lr=params["lr0"], weight_decay=params["weight_decay"])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=params["epochs"])
    criterion = nn.CrossEntropyLoss(label_smoothing=params["label_smoothing"])

    best_val_acc = 0.0
    epochs_sans_amelioration = 0
    historique = {"epoch": [], "train_loss": [], "val_loss": [], "val_acc": []}
    best_model_path = "/content/best_efficientnetb0.pt"

    for epoch in range(1, params["epochs"] + 1):
        t0 = time.time()
        model.train()
        train_loss = 0.0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * imgs.size(0)
        train_loss /= len(train_loader.dataset)

        model.eval()
        val_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                outputs = model(imgs)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * imgs.size(0)
                preds = outputs.argmax(dim=1)
                correct += (preds == labels).sum().item()
                total += labels.size(0)
        val_loss /= len(val_loader.dataset)
        val_acc = correct / total

        scheduler.step()
        dt = time.time() - t0
        print(f"Epoch {epoch}/{params['epochs']} - train_loss={train_loss:.4f} "
              f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} ({dt:.1f}s)")

        historique["epoch"].append(epoch)
        historique["train_loss"].append(train_loss)
        historique["val_loss"].append(val_loss)
        historique["val_acc"].append(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            epochs_sans_amelioration = 0
            torch.save(model.state_dict(), best_model_path)
        else:
            epochs_sans_amelioration += 1
            if epochs_sans_amelioration >= params["patience"]:
                print(f"\n[EarlyStopping] Arrêt à l'epoch {epoch} (meilleur val_acc={best_val_acc:.4f})")
                break

    return best_model_path, historique, best_val_acc

# ==================================================================
# 9) TEST FINAL
# ==================================================================
def tester(model, best_model_path, loader, classes, device):
    model.load_state_dict(torch.load(best_model_path))
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            outputs = model(imgs)
            preds = outputs.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())

    acc = accuracy_score(all_labels, all_preds)
    print(f"\nTest accuracy : {acc:.4f}")
    print(classification_report(all_labels, all_preds, target_names=classes))

    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(6, 5))
    plt.imshow(cm, cmap="Blues")
    plt.title("Matrice de confusion")
    plt.colorbar()
    plt.xticks(range(len(classes)), classes, rotation=45, ha="right")
    plt.yticks(range(len(classes)), classes)
    for i in range(len(classes)):
        for j in range(len(classes)):
            plt.text(j, i, cm[i, j], ha="center", va="center")
    plt.xlabel("Prédiction"); plt.ylabel("Réel"); plt.tight_layout()
    plt.savefig("/content/confusion_matrix_effnet.png")
    plt.close()

    rapport = classification_report(all_labels, all_preds, target_names=classes, output_dict=True)
    return acc, rapport


# ==================================================================
# 10) SAUVEGARDE DANS DRIVE
# ==================================================================
def sauvegarder_dans_drive(df_stats, params, historique, test_acc, rapport, best_model_path):
    if df_stats is not None and not df_stats.empty:
        df_stats.to_csv(DRIVE_BASE / "Dataset" / "analyse_dataset.csv", index=False)
    dist_png = Path("/content/distribution_classes_effnet.png")
    if dist_png.exists():
        shutil.copy2(dist_png, DRIVE_BASE / "Dataset" / "distribution_classes.png")

    params_propre = {k: str(v) for k, v in params.items()}
    with open(DRIVE_BASE / "Hyperparametres" / "hyperparametres_efficientnetb0.json", "w") as f:
        json.dump(params_propre, f, indent=2, ensure_ascii=False)

    shutil.copy2(best_model_path, DRIVE_BASE / "Modeles" / "best_efficientnetb0.pt")

    df_hist = pd.DataFrame(historique)
    df_hist.to_csv(DRIVE_BASE / "Resultats" / "results.csv", index=False)

    plt.figure(figsize=(8, 4))
    plt.plot(df_hist["epoch"], df_hist["train_loss"], label="train_loss")
    plt.plot(df_hist["epoch"], df_hist["val_loss"], label="val_loss")
    plt.legend(); plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title("Courbes de loss")
    plt.tight_layout()
    plt.savefig(DRIVE_BASE / "Resultats" / "results.png")
    plt.close()

    cm_png = Path("/content/confusion_matrix_effnet.png")
    if cm_png.exists():
        shutil.copy2(cm_png, DRIVE_BASE / "Resultats" / "confusion_matrix.png")

    with open(DRIVE_BASE / "Resultats" / "classification_report.json", "w") as f:
        json.dump(rapport, f, indent=2, ensure_ascii=False)

    readme = f"""# EfficientNetB0(2) — Classification

## Resultats
- Test accuracy : {test_acc:.4f}
- Voir Resultats/classification_report.json pour le détail par classe
"""
    with open(DRIVE_BASE / "README_EfficientNetB0.md", "w") as f:
        f.write(readme)

    print(f"\n✅ Tout est dans ton Drive : {DRIVE_BASE}")


# ==================================================================
# 11) EXECUTION COMPLETE
# ==================================================================
df_stats = analyser_dataset(DATASET_PATH)
clean_path = nettoyer_dataset(DATASET_PATH, WORK_PATH)
creer_val_si_absent(clean_path, ratio_val=0.2)

params = obtenir_best_params()
print(f"\n[Info] Device utilisé : {params['device']}")

train_loader, val_loader, test_loader, classes = preparer_dataloaders(clean_path, params)
print(f"[Info] Classes : {classes}")

model = construire_modele(len(classes), params["device"])
best_model_path, historique, best_val_acc = entrainer(model, train_loader, val_loader, params)

loader_test_final = test_loader if test_loader else val_loader
test_acc, rapport = tester(model, best_model_path, loader_test_final, classes, params["device"])

sauvegarder_dans_drive(df_stats, params, historique, test_acc, rapport, best_model_path)

In [ ]:
def tester(model, best_model_path, loader, classes, device):
    model.load_state_dict(torch.load(best_model_path))
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            outputs = model(imgs)
            preds = outputs.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())

    acc = accuracy_score(all_labels, all_preds)
    print(f"\nTest accuracy : {acc:.4f}")

    labels_indices = list(range(len(classes)))
    print(classification_report(all_labels, all_preds, labels=labels_indices,
                                 target_names=classes, zero_division=0))

    cm = confusion_matrix(all_labels, all_preds, labels=labels_indices)
    plt.figure(figsize=(6, 5))
    plt.imshow(cm, cmap="Blues")
    plt.title("Matrice de confusion")
    plt.colorbar()
    plt.xticks(range(len(classes)), classes, rotation=45, ha="right")
    plt.yticks(range(len(classes)), classes)
    for i in range(len(classes)):
        for j in range(len(classes)):
            plt.text(j, i, cm[i, j], ha="center", va="center")
    plt.xlabel("Prédiction"); plt.ylabel("Réel"); plt.tight_layout()
    plt.savefig("/content/confusion_matrix_effnet.png")
    plt.close()

    rapport = classification_report(all_labels, all_preds, labels=labels_indices,
                                     target_names=classes, output_dict=True, zero_division=0)
    return acc, rapport

In [ ]:
loader_test_final = test_loader if test_loader else val_loader
test_acc, rapport = tester(model, best_model_path, loader_test_final, classes, params["device"])
sauvegarder_dans_drive(df_stats, params, historique, test_acc, rapport, best_model_path)

In [5]:
"""
==========================================================
PIPELINE EfficientNetB2 - CLASSIFICATION (Google Colab, GPU)
==========================================================
Fine-tuning complet (pas de backbone gelé, contrairement à la
version CPU) — sauvegarde directement dans Drive/MonProjet/EfficientNetB2(2)/
==========================================================
"""

!pip install -q torch torchvision scikit-learn

import os, json, shutil, random, hashlib, time
from pathlib import Path
from collections import defaultdict

from PIL import Image, ImageFile
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

# ==================================================================
# 1) MONTAGE DRIVE
# ==================================================================
from google.colab import drive
drive.mount('/content/drive')

# ==================================================================
# 2) CONFIGURATION
# ==================================================================
DATASET_PATH = "/content/drive/MyDrive/MonProjet/NewDataSet/concrete-damage-v3"
WORK_PATH    = "/content/newdataset_clean_effnetb2"
IMG_SIZE     = 260   # résolution native EfficientNetB2
SPLITS       = ["train", "val", "test"]

DRIVE_BASE = Path("/content/drive/MyDrive/MonProjet/EfficientNetB2(2)")
for d in ["Dataset", "Hyperparametres", "Modeles", "Resultats"]:
    (DRIVE_BASE / d).mkdir(parents=True, exist_ok=True)

print(f"[Info] Dataset source : {DATASET_PATH}")
print(f"[Info] Existe ? {Path(DATASET_PATH).exists()}")
print(f"[Info] GPU disponible : {torch.cuda.is_available()}")

# ==================================================================
# 3) ANALYSE EXPLORATOIRE
# ==================================================================
def analyser_dataset(dataset_path):
    print("="*60); print("ANALYSE DU DATASET"); print("="*60)
    stats = []
    for split in SPLITS:
        split_path = Path(dataset_path) / split
        if not split_path.exists():
            print(f"[!] Dossier '{split}' introuvable, ignoré.")
            continue
        for cls_dir in [d for d in split_path.iterdir() if d.is_dir()]:
            images = list(cls_dir.glob("*.*"))
            widths, heights, sizes_ko = [], [], []
            for img_path in images[:200]:
                try:
                    with Image.open(img_path) as img:
                        w, h = img.size
                        widths.append(w); heights.append(h)
                    sizes_ko.append(os.path.getsize(img_path) / 1024)
                except Exception:
                    continue
            stats.append({
                "split": split, "classe": cls_dir.name, "n_images": len(images),
                "largeur_moy": sum(widths)/len(widths) if widths else None,
                "hauteur_moy": sum(heights)/len(heights) if heights else None,
                "poids_moy_ko": sum(sizes_ko)/len(sizes_ko) if sizes_ko else None,
            })
    df = pd.DataFrame(stats)
    print(df.to_string(index=False))
    if not df.empty:
        train_df = df[df["split"] == "train"]
        if not train_df.empty:
            plt.figure(figsize=(8, 4))
            plt.bar(train_df["classe"], train_df["n_images"])
            plt.xticks(rotation=45, ha="right")
            plt.title("Distribution des classes (train)")
            plt.ylabel("Nombre d'images")
            plt.tight_layout()
            plt.savefig("/content/distribution_classes_effnetb2.png")
            plt.close()
            ratio = train_df["n_images"].max() / train_df["n_images"].min()
            print(f"\nRatio classe max/min : {ratio:.2f}")
    return df

# ==================================================================
# 4) DATA CLEANING
# ==================================================================
def detecter_images_corrompues(dataset_path):
    corrompues = []
    for split in SPLITS:
        split_path = Path(dataset_path) / split
        if not split_path.exists():
            continue
        for img_path in split_path.rglob("*.*"):
            try:
                with Image.open(img_path) as img:
                    img.load()
            except Exception:
                corrompues.append(img_path)
    print(f"Images corrompues/tronquées détectées : {len(corrompues)}")
    return corrompues

def detecter_doublons(dataset_path):
    hashes = defaultdict(list)
    for split in SPLITS:
        split_path = Path(dataset_path) / split
        if not split_path.exists():
            continue
        for img_path in split_path.rglob("*.*"):
            try:
                with open(img_path, "rb") as f:
                    h = hashlib.md5(f.read()).hexdigest()
                hashes[h].append(img_path)
            except Exception:
                continue
    doublons = []
    for h, paths in hashes.items():
        if len(paths) > 1:
            doublons.extend(paths[1:])
    print(f"Doublons détectés : {len(doublons)}")
    return doublons

def nettoyer_dataset(dataset_path, work_path):
    corrompues = set(detecter_images_corrompues(dataset_path))
    doublons = set(detecter_doublons(dataset_path))
    a_exclure = corrompues | doublons
    if os.path.exists(work_path):
        shutil.rmtree(work_path)
    n_copiees, n_exclues = 0, 0
    for split in SPLITS:
        split_path = Path(dataset_path) / split
        if not split_path.exists():
            continue
        for img_path in split_path.rglob("*.*"):
            if img_path in a_exclure:
                n_exclues += 1
                continue
            rel = img_path.relative_to(dataset_path)
            dest = Path(work_path) / rel
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(img_path, dest)
            n_copiees += 1
    print(f"\nNettoyage terminé : {n_copiees} images copiées, {n_exclues} exclues.")
    return work_path

def creer_val_si_absent(work_path, ratio_val=0.2, seed=42):
    random.seed(seed)
    train_path = Path(work_path) / "train"
    val_path = Path(work_path) / "val"
    if val_path.exists() and any(val_path.iterdir()):
        print("[Split] 'val' déjà présent.")
        return
    if not train_path.exists():
        return
    print(f"[Split] Création automatique de val ({ratio_val:.0%} de train).")
    for cls_dir in train_path.iterdir():
        if not cls_dir.is_dir():
            continue
        images = list(cls_dir.glob("*.*"))
        random.shuffle(images)
        n_val = max(1, int(len(images) * ratio_val))
        dest_dir = val_path / cls_dir.name
        dest_dir.mkdir(parents=True, exist_ok=True)
        for img in images[:n_val]:
            shutil.move(str(img), str(dest_dir / img.name))

# ==================================================================
# 5) MEILLEURS HYPERPARAMÈTRES (EfficientNetB2, fine-tuning complet)
# ==================================================================
def obtenir_best_params():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    return dict(
        model="efficientnet_b2 (ImageNet pretrained)",
        imgsz=IMG_SIZE,
        epochs=30,
        batch=24,
        optimizer="AdamW",
        lr0=0.0003,
        weight_decay=1e-4,
        scheduler="CosineAnnealingLR",
        label_smoothing=0.1,
        patience=7,
        dropout=0.3,
        freeze_backbone=False,
        augment=True,
        device=device,
        seed=42,
    )

# ==================================================================
# 6) DATALOADERS
# ==================================================================
def preparer_dataloaders(work_path, params):
    mean = [0.485, 0.456, 0.406]
    std  = [0.229, 0.224, 0.225]

    train_tf = transforms.Compose([
        transforms.RandomResizedCrop(params["imgsz"], scale=(0.8, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])
    eval_tf = transforms.Compose([
        transforms.Resize(int(params["imgsz"] * 1.14)),
        transforms.CenterCrop(params["imgsz"]),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])

    train_ds = datasets.ImageFolder(str(Path(work_path) / "train"), transform=train_tf)
    val_ds   = datasets.ImageFolder(str(Path(work_path) / "val"), transform=eval_tf)

    test_path = Path(work_path) / "test"
    test_ds = datasets.ImageFolder(str(test_path), transform=eval_tf) if test_path.exists() else None

    train_loader = DataLoader(train_ds, batch_size=params["batch"], shuffle=True, num_workers=2)
    val_loader   = DataLoader(val_ds, batch_size=params["batch"], shuffle=False, num_workers=2)
    test_loader  = DataLoader(test_ds, batch_size=params["batch"], shuffle=False, num_workers=2) if test_ds else None

    return train_loader, val_loader, test_loader, train_ds.classes

# ==================================================================
# 7) MODELE
# ==================================================================
def construire_modele(num_classes, params):
    model = models.efficientnet_b2(weights=models.EfficientNet_B2_Weights.IMAGENET1K_V1)
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)
    return model.to(params["device"])

# ==================================================================
# 8) ENTRAINEMENT AVEC EARLY STOPPING
# ==================================================================
def entrainer(model, train_loader, val_loader, params):
    device = params["device"]
    optimizer = torch.optim.AdamW(model.parameters(), lr=params["lr0"], weight_decay=params["weight_decay"])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=params["epochs"])
    criterion = nn.CrossEntropyLoss(label_smoothing=params["label_smoothing"])

    best_val_acc = 0.0
    epochs_sans_amelioration = 0
    historique = {"epoch": [], "train_loss": [], "val_loss": [], "val_acc": []}
    best_model_path = "/content/best_efficientnetb2.pt"

    for epoch in range(1, params["epochs"] + 1):
        t0 = time.time()
        model.train()
        train_loss = 0.0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * imgs.size(0)
        train_loss /= len(train_loader.dataset)

        model.eval()
        val_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                outputs = model(imgs)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * imgs.size(0)
                preds = outputs.argmax(dim=1)
                correct += (preds == labels).sum().item()
                total += labels.size(0)
        val_loss /= len(val_loader.dataset)
        val_acc = correct / total

        scheduler.step()
        dt = time.time() - t0
        print(f"Epoch {epoch}/{params['epochs']} - train_loss={train_loss:.4f} "
              f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} ({dt:.1f}s)")

        historique["epoch"].append(epoch)
        historique["train_loss"].append(train_loss)
        historique["val_loss"].append(val_loss)
        historique["val_acc"].append(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            epochs_sans_amelioration = 0
            torch.save(model.state_dict(), best_model_path)
        else:
            epochs_sans_amelioration += 1
            if epochs_sans_amelioration >= params["patience"]:
                print(f"\n[EarlyStopping] Arrêt à l'epoch {epoch} (meilleur val_acc={best_val_acc:.4f})")
                break

    return best_model_path, historique, best_val_acc

# ==================================================================
# 9) TEST FINAL
# ==================================================================
def tester(model, best_model_path, loader, classes, device):
    model.load_state_dict(torch.load(best_model_path))
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            outputs = model(imgs)
            preds = outputs.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())

    acc = accuracy_score(all_labels, all_preds)
    print(f"\nTest accuracy : {acc:.4f}")

    labels_indices = list(range(len(classes)))
    print(classification_report(all_labels, all_preds, labels=labels_indices,
                                 target_names=classes, zero_division=0))

    cm = confusion_matrix(all_labels, all_preds, labels=labels_indices)
    plt.figure(figsize=(6, 5))
    plt.imshow(cm, cmap="Blues")
    plt.title("Matrice de confusion")
    plt.colorbar()
    plt.xticks(range(len(classes)), classes, rotation=45, ha="right")
    plt.yticks(range(len(classes)), classes)
    for i in range(len(classes)):
        for j in range(len(classes)):
            plt.text(j, i, cm[i, j], ha="center", va="center")
    plt.xlabel("Prédiction"); plt.ylabel("Réel"); plt.tight_layout()
    plt.savefig("/content/confusion_matrix_effnetb2.png")
    plt.close()

    rapport = classification_report(all_labels, all_preds, labels=labels_indices,
                                     target_names=classes, output_dict=True, zero_division=0)
    return acc, rapport

# ==================================================================
# 10) SAUVEGARDE DANS DRIVE
# ==================================================================
def sauvegarder_dans_drive(df_stats, params, historique, test_acc, rapport, best_model_path):
    if df_stats is not None and not df_stats.empty:
        df_stats.to_csv(DRIVE_BASE / "Dataset" / "analyse_dataset.csv", index=False)
    dist_png = Path("/content/distribution_classes_effnetb2.png")
    if dist_png.exists():
        shutil.copy2(dist_png, DRIVE_BASE / "Dataset" / "distribution_classes.png")

    params_propre = {k: str(v) for k, v in params.items()}
    with open(DRIVE_BASE / "Hyperparametres" / "hyperparametres_efficientnetb2.json", "w") as f:
        json.dump(params_propre, f, indent=2, ensure_ascii=False)

    shutil.copy2(best_model_path, DRIVE_BASE / "Modeles" / "best_efficientnetb2.pt")

    df_hist = pd.DataFrame(historique)
    df_hist.to_csv(DRIVE_BASE / "Resultats" / "results.csv", index=False)

    plt.figure(figsize=(8, 4))
    plt.plot(df_hist["epoch"], df_hist["train_loss"], label="train_loss")
    plt.plot(df_hist["epoch"], df_hist["val_loss"], label="val_loss")
    plt.legend(); plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title("Courbes de loss")
    plt.tight_layout()
    plt.savefig(DRIVE_BASE / "Resultats" / "results.png")
    plt.close()

    cm_png = Path("/content/confusion_matrix_effnetb2.png")
    if cm_png.exists():
        shutil.copy2(cm_png, DRIVE_BASE / "Resultats" / "confusion_matrix.png")

    with open(DRIVE_BASE / "Resultats" / "classification_report.json", "w") as f:
        json.dump(rapport, f, indent=2, ensure_ascii=False)

    readme = f"""# EfficientNetB2(2) — Classification

## Resultats
- Test accuracy : {test_acc:.4f}
- Voir Resultats/classification_report.json pour le détail par classe
"""
    with open(DRIVE_BASE / "README_EfficientNetB2.md", "w") as f:
        f.write(readme)

    print(f"\n✅ Tout est dans ton Drive : {DRIVE_BASE}")


# ==================================================================
# 11) EXECUTION COMPLETE
# ==================================================================
df_stats = analyser_dataset(DATASET_PATH)
clean_path = nettoyer_dataset(DATASET_PATH, WORK_PATH)
creer_val_si_absent(clean_path, ratio_val=0.2)

params = obtenir_best_params()
print(f"\n[Info] Device utilisé : {params['device']}")

ImageFile.LOAD_TRUNCATED_IMAGES = True

train_loader, val_loader, test_loader, classes = preparer_dataloaders(clean_path, params)
print(f"[Info] Classes : {classes}")

model = construire_modele(len(classes), params)
best_model_path, historique, best_val_acc = entrainer(model, train_loader, val_loader, params)

loader_test_final = test_loader if test_loader else val_loader
test_acc, rapport = tester(model, best_model_path, loader_test_final, classes, params["device"])

sauvegarder_dans_drive(df_stats, params, historique, test_acc, rapport, best_model_path)

MessageError: Error: credential propagation was unsuccessful

In [ ]:
"""
==========================================================
PIPELINE ResNet50 - CLASSIFICATION (Google Colab, GPU)
==========================================================
Fine-tuning complet — sauvegarde directement dans
Drive/MonProjet/ResNet50(2)/
==========================================================
"""

!pip install -q torch torchvision scikit-learn

import os, json, shutil, random, hashlib, time
from pathlib import Path
from collections import defaultdict

from PIL import Image, ImageFile
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

# ==================================================================
# 1) MONTAGE DRIVE
# ==================================================================
from google.colab import drive
drive.mount('/content/drive')

# ==================================================================
# 2) CONFIGURATION
# ==================================================================
DATASET_PATH = "/content/drive/MyDrive/MonProjet/NewDataSet/concrete-damage-v3"
WORK_PATH    = "/content/newdataset_clean_resnet50"
IMG_SIZE     = 224   # résolution standard ResNet
SPLITS       = ["train", "val", "test"]

DRIVE_BASE = Path("/content/drive/MyDrive/MonProjet/ResNet50(2)")
for d in ["Dataset", "Hyperparametres", "Modeles", "Resultats"]:
    (DRIVE_BASE / d).mkdir(parents=True, exist_ok=True)

print(f"[Info] Dataset source : {DATASET_PATH}")
print(f"[Info] Existe ? {Path(DATASET_PATH).exists()}")
print(f"[Info] GPU disponible : {torch.cuda.is_available()}")

# ==================================================================
# 3) ANALYSE EXPLORATOIRE
# ==================================================================
def analyser_dataset(dataset_path):
    print("="*60); print("ANALYSE DU DATASET"); print("="*60)
    stats = []
    for split in SPLITS:
        split_path = Path(dataset_path) / split
        if not split_path.exists():
            print(f"[!] Dossier '{split}' introuvable, ignoré.")
            continue
        for cls_dir in [d for d in split_path.iterdir() if d.is_dir()]:
            images = list(cls_dir.glob("*.*"))
            widths, heights, sizes_ko = [], [], []
            for img_path in images[:200]:
                try:
                    with Image.open(img_path) as img:
                        w, h = img.size
                        widths.append(w); heights.append(h)
                    sizes_ko.append(os.path.getsize(img_path) / 1024)
                except Exception:
                    continue
            stats.append({
                "split": split, "classe": cls_dir.name, "n_images": len(images),
                "largeur_moy": sum(widths)/len(widths) if widths else None,
                "hauteur_moy": sum(heights)/len(heights) if heights else None,
                "poids_moy_ko": sum(sizes_ko)/len(sizes_ko) if sizes_ko else None,
            })
    df = pd.DataFrame(stats)
    print(df.to_string(index=False))
    if not df.empty:
        train_df = df[df["split"] == "train"]
        if not train_df.empty:
            plt.figure(figsize=(8, 4))
            plt.bar(train_df["classe"], train_df["n_images"])
            plt.xticks(rotation=45, ha="right")
            plt.title("Distribution des classes (train)")
            plt.ylabel("Nombre d'images")
            plt.tight_layout()
            plt.savefig("/content/distribution_classes_resnet50.png")
            plt.close()
            ratio = train_df["n_images"].max() / train_df["n_images"].min()
            print(f"\nRatio classe max/min : {ratio:.2f}")
    return df

# ==================================================================
# 4) DATA CLEANING
# ==================================================================
def detecter_images_corrompues(dataset_path):
    corrompues = []
    for split in SPLITS:
        split_path = Path(dataset_path) / split
        if not split_path.exists():
            continue
        for img_path in split_path.rglob("*.*"):
            try:
                with Image.open(img_path) as img:
                    img.load()
            except Exception:
                corrompues.append(img_path)
    print(f"Images corrompues/tronquées détectées : {len(corrompues)}")
    return corrompues

def detecter_doublons(dataset_path):
    hashes = defaultdict(list)
    for split in SPLITS:
        split_path = Path(dataset_path) / split
        if not split_path.exists():
            continue
        for img_path in split_path.rglob("*.*"):
            try:
                with open(img_path, "rb") as f:
                    h = hashlib.md5(f.read()).hexdigest()
                hashes[h].append(img_path)
            except Exception:
                continue
    doublons = []
    for h, paths in hashes.items():
        if len(paths) > 1:
            doublons.extend(paths[1:])
    print(f"Doublons détectés : {len(doublons)}")
    return doublons

def nettoyer_dataset(dataset_path, work_path):
    corrompues = set(detecter_images_corrompues(dataset_path))
    doublons = set(detecter_doublons(dataset_path))
    a_exclure = corrompues | doublons
    if os.path.exists(work_path):
        shutil.rmtree(work_path)
    n_copiees, n_exclues = 0, 0
    for split in SPLITS:
        split_path = Path(dataset_path) / split
        if not split_path.exists():
            continue
        for img_path in split_path.rglob("*.*"):
            if img_path in a_exclure:
                n_exclues += 1
                continue
            rel = img_path.relative_to(dataset_path)
            dest = Path(work_path) / rel
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(img_path, dest)
            n_copiees += 1
    print(f"\nNettoyage terminé : {n_copiees} images copiées, {n_exclues} exclues.")
    return work_path

def creer_val_si_absent(work_path, ratio_val=0.2, seed=42):
    random.seed(seed)
    train_path = Path(work_path) / "train"
    val_path = Path(work_path) / "val"
    if val_path.exists() and any(val_path.iterdir()):
        print("[Split] 'val' déjà présent.")
        return
    if not train_path.exists():
        return
    print(f"[Split] Création automatique de val ({ratio_val:.0%} de train).")
    for cls_dir in train_path.iterdir():
        if not cls_dir.is_dir():
            continue
        images = list(cls_dir.glob("*.*"))
        random.shuffle(images)
        n_val = max(1, int(len(images) * ratio_val))
        dest_dir = val_path / cls_dir.name
        dest_dir.mkdir(parents=True, exist_ok=True)
        for img in images[:n_val]:
            shutil.move(str(img), str(dest_dir / img.name))

# ==================================================================
# 5) MEILLEURS HYPERPARAMÈTRES (ResNet50, fine-tuning complet)
# ==================================================================
def obtenir_best_params():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    return dict(
        model="resnet50 (ImageNet pretrained)",
        imgsz=IMG_SIZE,
        epochs=30,
        batch=32,                  # ResNet50 (25M params) tient bien un batch de 32 sur T4
        optimizer="AdamW",
        lr0=0.0003,
        weight_decay=1e-4,
        scheduler="CosineAnnealingLR",
        label_smoothing=0.1,
        patience=7,
        augment=True,
        device=device,
        seed=42,
    )

# ==================================================================
# 6) DATALOADERS
# ==================================================================
def preparer_dataloaders(work_path, params):
    mean = [0.485, 0.456, 0.406]
    std  = [0.229, 0.224, 0.225]

    train_tf = transforms.Compose([
        transforms.RandomResizedCrop(params["imgsz"], scale=(0.8, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])
    eval_tf = transforms.Compose([
        transforms.Resize(int(params["imgsz"] * 1.14)),
        transforms.CenterCrop(params["imgsz"]),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])

    train_ds = datasets.ImageFolder(str(Path(work_path) / "train"), transform=train_tf)
    val_ds   = datasets.ImageFolder(str(Path(work_path) / "val"), transform=eval_tf)

    test_path = Path(work_path) / "test"
    test_ds = datasets.ImageFolder(str(test_path), transform=eval_tf) if test_path.exists() else None

    train_loader = DataLoader(train_ds, batch_size=params["batch"], shuffle=True, num_workers=2)
    val_loader   = DataLoader(val_ds, batch_size=params["batch"], shuffle=False, num_workers=2)
    test_loader  = DataLoader(test_ds, batch_size=params["batch"], shuffle=False, num_workers=2) if test_ds else None

    return train_loader, val_loader, test_loader, train_ds.classes

# ==================================================================
# 7) MODELE
# ==================================================================
def construire_modele(num_classes, params):
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)
    return model.to(params["device"])

# ==================================================================
# 8) ENTRAINEMENT AVEC EARLY STOPPING
# ==================================================================
def entrainer(model, train_loader, val_loader, params):
    device = params["device"]
    optimizer = torch.optim.AdamW(model.parameters(), lr=params["lr0"], weight_decay=params["weight_decay"])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=params["epochs"])
    criterion = nn.CrossEntropyLoss(label_smoothing=params["label_smoothing"])

    best_val_acc = 0.0
    epochs_sans_amelioration = 0
    historique = {"epoch": [], "train_loss": [], "val_loss": [], "val_acc": []}
    best_model_path = "/content/best_resnet50.pt"

    for epoch in range(1, params["epochs"] + 1):
        t0 = time.time()
        model.train()
        train_loss = 0.0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * imgs.size(0)
        train_loss /= len(train_loader.dataset)

        model.eval()
        val_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                outputs = model(imgs)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * imgs.size(0)
                preds = outputs.argmax(dim=1)
                correct += (preds == labels).sum().item()
                total += labels.size(0)
        val_loss /= len(val_loader.dataset)
        val_acc = correct / total

        scheduler.step()
        dt = time.time() - t0
        print(f"Epoch {epoch}/{params['epochs']} - train_loss={train_loss:.4f} "
              f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} ({dt:.1f}s)")

        historique["epoch"].append(epoch)
        historique["train_loss"].append(train_loss)
        historique["val_loss"].append(val_loss)
        historique["val_acc"].append(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            epochs_sans_amelioration = 0
            torch.save(model.state_dict(), best_model_path)
        else:
            epochs_sans_amelioration += 1
            if epochs_sans_amelioration >= params["patience"]:
                print(f"\n[EarlyStopping] Arrêt à l'epoch {epoch} (meilleur val_acc={best_val_acc:.4f})")
                break

    return best_model_path, historique, best_val_acc

# ==================================================================
# 9) TEST FINAL
# ==================================================================
def tester(model, best_model_path, loader, classes, device):
    model.load_state_dict(torch.load(best_model_path))
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            outputs = model(imgs)
            preds = outputs.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())

    acc = accuracy_score(all_labels, all_preds)
    print(f"\nTest accuracy : {acc:.4f}")

    labels_indices = list(range(len(classes)))
    print(classification_report(all_labels, all_preds, labels=labels_indices,
                                 target_names=classes, zero_division=0))

    cm = confusion_matrix(all_labels, all_preds, labels=labels_indices)
    plt.figure(figsize=(6, 5))
    plt.imshow(cm, cmap="Blues")
    plt.title("Matrice de confusion")
    plt.colorbar()
    plt.xticks(range(len(classes)), classes, rotation=45, ha="right")
    plt.yticks(range(len(classes)), classes)
    for i in range(len(classes)):
        for j in range(len(classes)):
            plt.text(j, i, cm[i, j], ha="center", va="center")
    plt.xlabel("Prédiction"); plt.ylabel("Réel"); plt.tight_layout()
    plt.savefig("/content/confusion_matrix_resnet50.png")
    plt.close()

    rapport = classification_report(all_labels, all_preds, labels=labels_indices,
                                     target_names=classes, output_dict=True, zero_division=0)
    return acc, rapport

# ==================================================================
# 10) SAUVEGARDE DANS DRIVE
# ==================================================================
def sauvegarder_dans_drive(df_stats, params, historique, test_acc, rapport, best_model_path):
    if df_stats is not None and not df_stats.empty:
        df_stats.to_csv(DRIVE_BASE / "Dataset" / "analyse_dataset.csv", index=False)
    dist_png = Path("/content/distribution_classes_resnet50.png")
    if dist_png.exists():
        shutil.copy2(dist_png, DRIVE_BASE / "Dataset" / "distribution_classes.png")

    params_propre = {k: str(v) for k, v in params.items()}
    with open(DRIVE_BASE / "Hyperparametres" / "hyperparametres_resnet50.json", "w") as f:
        json.dump(params_propre, f, indent=2, ensure_ascii=False)

    shutil.copy2(best_model_path, DRIVE_BASE / "Modeles" / "best_resnet50.pt")

    df_hist = pd.DataFrame(historique)
    df_hist.to_csv(DRIVE_BASE / "Resultats" / "results.csv", index=False)

    plt.figure(figsize=(8, 4))
    plt.plot(df_hist["epoch"], df_hist["train_loss"], label="train_loss")
    plt.plot(df_hist["epoch"], df_hist["val_loss"], label="val_loss")
    plt.legend(); plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title("Courbes de loss")
    plt.tight_layout()
    plt.savefig(DRIVE_BASE / "Resultats" / "results.png")
    plt.close()

    cm_png = Path("/content/confusion_matrix_resnet50.png")
    if cm_png.exists():
        shutil.copy2(cm_png, DRIVE_BASE / "Resultats" / "confusion_matrix.png")

    with open(DRIVE_BASE / "Resultats" / "classification_report.json", "w") as f:
        json.dump(rapport, f, indent=2, ensure_ascii=False)

    readme = f"""# ResNet50(2) — Classification

## Resultats
- Test accuracy : {test_acc:.4f}
- Voir Resultats/classification_report.json pour le détail par classe
"""
    with open(DRIVE_BASE / "README_ResNet50.md", "w") as f:
        f.write(readme)

    print(f"\n✅ Tout est dans ton Drive : {DRIVE_BASE}")


# ==================================================================
# 11) EXECUTION COMPLETE
# ==================================================================
df_stats = analyser_dataset(DATASET_PATH)
clean_path = nettoyer_dataset(DATASET_PATH, WORK_PATH)
creer_val_si_absent(clean_path, ratio_val=0.2)

params = obtenir_best_params()
print(f"\n[Info] Device utilisé : {params['device']}")

ImageFile.LOAD_TRUNCATED_IMAGES = True

train_loader, val_loader, test_loader, classes = preparer_dataloaders(clean_path, params)
print(f"[Info] Classes : {classes}")

model = construire_modele(len(classes), params)
best_model_path, historique, best_val_acc = entrainer(model, train_loader, val_loader, params)

loader_test_final = test_loader if test_loader else val_loader
test_acc, rapport = tester(model, best_model_path, loader_test_final, classes, params["device"])

sauvegarder_dans_drive(df_stats, params, historique, test_acc, rapport, best_model_path)

In [ ]:
"""
==========================================================
PIPELINE ResNet50 - CLASSIFICATION (Google Colab, GPU)
==========================================================
Fine-tuning complet — sauvegarde directement dans
Drive/MonProjet/ResNet50(2)/
==========================================================
"""

!pip install -q torch torchvision scikit-learn

import os, json, shutil, random, hashlib, time
from pathlib import Path
from collections import defaultdict

from PIL import Image, ImageFile
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

# ==================================================================
# 1) MONTAGE DRIVE
# ==================================================================
from google.colab import drive
drive.mount('/content/drive')

# ==================================================================
# 2) CONFIGURATION
# ==================================================================
DATASET_PATH = "/content/drive/MyDrive/MonProjet/NewDataSet/concrete-damage-v3"
WORK_PATH    = "/content/newdataset_clean_resnet50"
IMG_SIZE     = 224   # résolution standard ResNet
SPLITS       = ["train", "val", "test"]

DRIVE_BASE = Path("/content/drive/MyDrive/MonProjet/ResNet50(2)")
for d in ["Dataset", "Hyperparametres", "Modeles", "Resultats"]:
    (DRIVE_BASE / d).mkdir(parents=True, exist_ok=True)

print(f"[Info] Dataset source : {DATASET_PATH}")
print(f"[Info] Existe ? {Path(DATASET_PATH).exists()}")
print(f"[Info] GPU disponible : {torch.cuda.is_available()}")

# ==================================================================
# 3) ANALYSE EXPLORATOIRE
# ==================================================================
def analyser_dataset(dataset_path):
    print("="*60); print("ANALYSE DU DATASET"); print("="*60)
    stats = []
    for split in SPLITS:
        split_path = Path(dataset_path) / split
        if not split_path.exists():
            print(f"[!] Dossier '{split}' introuvable, ignoré.")
            continue
        for cls_dir in [d for d in split_path.iterdir() if d.is_dir()]:
            images = list(cls_dir.glob("*.*"))
            widths, heights, sizes_ko = [], [], []
            for img_path in images[:200]:
                try:
                    with Image.open(img_path) as img:
                        w, h = img.size
                        widths.append(w); heights.append(h)
                    sizes_ko.append(os.path.getsize(img_path) / 1024)
                except Exception:
                    continue
            stats.append({
                "split": split, "classe": cls_dir.name, "n_images": len(images),
                "largeur_moy": sum(widths)/len(widths) if widths else None,
                "hauteur_moy": sum(heights)/len(heights) if heights else None,
                "poids_moy_ko": sum(sizes_ko)/len(sizes_ko) if sizes_ko else None,
            })
    df = pd.DataFrame(stats)
    print(df.to_string(index=False))
    if not df.empty:
        train_df = df[df["split"] == "train"]
        if not train_df.empty:
            plt.figure(figsize=(8, 4))
            plt.bar(train_df["classe"], train_df["n_images"])
            plt.xticks(rotation=45, ha="right")
            plt.title("Distribution des classes (train)")
            plt.ylabel("Nombre d'images")
            plt.tight_layout()
            plt.savefig("/content/distribution_classes_resnet50.png")
            plt.close()
            ratio = train_df["n_images"].max() / train_df["n_images"].min()
            print(f"\nRatio classe max/min : {ratio:.2f}")
    return df

# ==================================================================
# 4) DATA CLEANING
# ==================================================================
def detecter_images_corrompues(dataset_path):
    corrompues = []
    for split in SPLITS:
        split_path = Path(dataset_path) / split
        if not split_path.exists():
            continue
        for img_path in split_path.rglob("*.*"):
            try:
                with Image.open(img_path) as img:
                    img.load()
            except Exception:
                corrompues.append(img_path)
    print(f"Images corrompues/tronquées détectées : {len(corrompues)}")
    return corrompues

def detecter_doublons(dataset_path):
    hashes = defaultdict(list)
    for split in SPLITS:
        split_path = Path(dataset_path) / split
        if not split_path.exists():
            continue
        for img_path in split_path.rglob("*.*"):
            try:
                with open(img_path, "rb") as f:
                    h = hashlib.md5(f.read()).hexdigest()
                hashes[h].append(img_path)
            except Exception:
                continue
    doublons = []
    for h, paths in hashes.items():
        if len(paths) > 1:
            doublons.extend(paths[1:])
    print(f"Doublons détectés : {len(doublons)}")
    return doublons

def nettoyer_dataset(dataset_path, work_path):
    corrompues = set(detecter_images_corrompues(dataset_path))
    doublons = set(detecter_doublons(dataset_path))
    a_exclure = corrompues | doublons
    if os.path.exists(work_path):
        shutil.rmtree(work_path)
    n_copiees, n_exclues = 0, 0
    for split in SPLITS:
        split_path = Path(dataset_path) / split
        if not split_path.exists():
            continue
        for img_path in split_path.rglob("*.*"):
            if img_path in a_exclure:
                n_exclues += 1
                continue
            rel = img_path.relative_to(dataset_path)
            dest = Path(work_path) / rel
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(img_path, dest)
            n_copiees += 1
    print(f"\nNettoyage terminé : {n_copiees} images copiées, {n_exclues} exclues.")
    return work_path

def creer_val_si_absent(work_path, ratio_val=0.2, seed=42):
    random.seed(seed)
    train_path = Path(work_path) / "train"
    val_path = Path(work_path) / "val"
    if val_path.exists() and any(val_path.iterdir()):
        print("[Split] 'val' déjà présent.")
        return
    if not train_path.exists():
        return
    print(f"[Split] Création automatique de val ({ratio_val:.0%} de train).")
    for cls_dir in train_path.iterdir():
        if not cls_dir.is_dir():
            continue
        images = list(cls_dir.glob("*.*"))
        random.shuffle(images)
        n_val = max(1, int(len(images) * ratio_val))
        dest_dir = val_path / cls_dir.name
        dest_dir.mkdir(parents=True, exist_ok=True)
        for img in images[:n_val]:
            shutil.move(str(img), str(dest_dir / img.name))

# ==================================================================
# 5) MEILLEURS HYPERPARAMÈTRES (ResNet50, fine-tuning complet)
# ==================================================================
def obtenir_best_params():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    return dict(
        model="resnet50 (ImageNet pretrained)",
        imgsz=IMG_SIZE,
        epochs=30,
        batch=32,                  # ResNet50 (25M params) tient bien un batch de 32 sur T4
        optimizer="AdamW",
        lr0=0.0003,
        weight_decay=1e-4,
        scheduler="CosineAnnealingLR",
        label_smoothing=0.1,
        patience=7,
        augment=True,
        device=device,
        seed=42,
    )

# ==================================================================
# 6) DATALOADERS
# ==================================================================
def preparer_dataloaders(work_path, params):
    mean = [0.485, 0.456, 0.406]
    std  = [0.229, 0.224, 0.225]

    train_tf = transforms.Compose([
        transforms.RandomResizedCrop(params["imgsz"], scale=(0.8, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])
    eval_tf = transforms.Compose([
        transforms.Resize(int(params["imgsz"] * 1.14)),
        transforms.CenterCrop(params["imgsz"]),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])

    train_ds = datasets.ImageFolder(str(Path(work_path) / "train"), transform=train_tf)
    val_ds   = datasets.ImageFolder(str(Path(work_path) / "val"), transform=eval_tf)

    test_path = Path(work_path) / "test"
    test_ds = datasets.ImageFolder(str(test_path), transform=eval_tf) if test_path.exists() else None

    train_loader = DataLoader(train_ds, batch_size=params["batch"], shuffle=True, num_workers=2)
    val_loader   = DataLoader(val_ds, batch_size=params["batch"], shuffle=False, num_workers=2)
    test_loader  = DataLoader(test_ds, batch_size=params["batch"], shuffle=False, num_workers=2) if test_ds else None

    return train_loader, val_loader, test_loader, train_ds.classes

# ==================================================================
# 7) MODELE
# ==================================================================
def construire_modele(num_classes, params):
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)
    return model.to(params["device"])

# ==================================================================
# 8) ENTRAINEMENT AVEC EARLY STOPPING
# ==================================================================
def entrainer(model, train_loader, val_loader, params):
    device = params["device"]
    optimizer = torch.optim.AdamW(model.parameters(), lr=params["lr0"], weight_decay=params["weight_decay"])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=params["epochs"])
    criterion = nn.CrossEntropyLoss(label_smoothing=params["label_smoothing"])

    best_val_acc = 0.0
    epochs_sans_amelioration = 0
    historique = {"epoch": [], "train_loss": [], "val_loss": [], "val_acc": []}
    best_model_path = "/content/best_resnet50.pt"

    for epoch in range(1, params["epochs"] + 1):
        t0 = time.time()
        model.train()
        train_loss = 0.0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * imgs.size(0)
        train_loss /= len(train_loader.dataset)

        model.eval()
        val_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                outputs = model(imgs)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * imgs.size(0)
                preds = outputs.argmax(dim=1)
                correct += (preds == labels).sum().item()
                total += labels.size(0)
        val_loss /= len(val_loader.dataset)
        val_acc = correct / total

        scheduler.step()
        dt = time.time() - t0
        print(f"Epoch {epoch}/{params['epochs']} - train_loss={train_loss:.4f} "
              f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} ({dt:.1f}s)")

        historique["epoch"].append(epoch)
        historique["train_loss"].append(train_loss)
        historique["val_loss"].append(val_loss)
        historique["val_acc"].append(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            epochs_sans_amelioration = 0
            torch.save(model.state_dict(), best_model_path)
        else:
            epochs_sans_amelioration += 1
            if epochs_sans_amelioration >= params["patience"]:
                print(f"\n[EarlyStopping] Arrêt à l'epoch {epoch} (meilleur val_acc={best_val_acc:.4f})")
                break

    return best_model_path, historique, best_val_acc

# ==================================================================
# 9) TEST FINAL
# ==================================================================
def tester(model, best_model_path, loader, classes, device):
    model.load_state_dict(torch.load(best_model_path))
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            outputs = model(imgs)
            preds = outputs.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())

    acc = accuracy_score(all_labels, all_preds)
    print(f"\nTest accuracy : {acc:.4f}")

    labels_indices = list(range(len(classes)))
    print(classification_report(all_labels, all_preds, labels=labels_indices,
                                 target_names=classes, zero_division=0))

    cm = confusion_matrix(all_labels, all_preds, labels=labels_indices)
    plt.figure(figsize=(6, 5))
    plt.imshow(cm, cmap="Blues")
    plt.title("Matrice de confusion")
    plt.colorbar()
    plt.xticks(range(len(classes)), classes, rotation=45, ha="right")
    plt.yticks(range(len(classes)), classes)
    for i in range(len(classes)):
        for j in range(len(classes)):
            plt.text(j, i, cm[i, j], ha="center", va="center")
    plt.xlabel("Prédiction"); plt.ylabel("Réel"); plt.tight_layout()
    plt.savefig("/content/confusion_matrix_resnet50.png")
    plt.close()

    rapport = classification_report(all_labels, all_preds, labels=labels_indices,
                                     target_names=classes, output_dict=True, zero_division=0)
    return acc, rapport

# ==================================================================
# 10) SAUVEGARDE DANS DRIVE
# ==================================================================
def sauvegarder_dans_drive(df_stats, params, historique, test_acc, rapport, best_model_path):
    if df_stats is not None and not df_stats.empty:
        df_stats.to_csv(DRIVE_BASE / "Dataset" / "analyse_dataset.csv", index=False)
    dist_png = Path("/content/distribution_classes_resnet50.png")
    if dist_png.exists():
        shutil.copy2(dist_png, DRIVE_BASE / "Dataset" / "distribution_classes.png")

    params_propre = {k: str(v) for k, v in params.items()}
    with open(DRIVE_BASE / "Hyperparametres" / "hyperparametres_resnet50.json", "w") as f:
        json.dump(params_propre, f, indent=2, ensure_ascii=False)

    shutil.copy2(best_model_path, DRIVE_BASE / "Modeles" / "best_resnet50.pt")

    df_hist = pd.DataFrame(historique)
    df_hist.to_csv(DRIVE_BASE / "Resultats" / "results.csv", index=False)

    plt.figure(figsize=(8, 4))
    plt.plot(df_hist["epoch"], df_hist["train_loss"], label="train_loss")
    plt.plot(df_hist["epoch"], df_hist["val_loss"], label="val_loss")
    plt.legend(); plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title("Courbes de loss")
    plt.tight_layout()
    plt.savefig(DRIVE_BASE / "Resultats" / "results.png")
    plt.close()

    cm_png = Path("/content/confusion_matrix_resnet50.png")
    if cm_png.exists():
        shutil.copy2(cm_png, DRIVE_BASE / "Resultats" / "confusion_matrix.png")

    with open(DRIVE_BASE / "Resultats" / "classification_report.json", "w") as f:
        json.dump(rapport, f, indent=2, ensure_ascii=False)

    readme = f"""# ResNet50(2) — Classification

## Resultats
- Test accuracy : {test_acc:.4f}
- Voir Resultats/classification_report.json pour le détail par classe
"""
    with open(DRIVE_BASE / "README_ResNet50.md", "w") as f:
        f.write(readme)

    print(f"\n✅ Tout est dans ton Drive : {DRIVE_BASE}")


# ==================================================================
# 11) EXECUTION COMPLETE
# ==================================================================
df_stats = analyser_dataset(DATASET_PATH)
clean_path = nettoyer_dataset(DATASET_PATH, WORK_PATH)
creer_val_si_absent(clean_path, ratio_val=0.2)

params = obtenir_best_params()
print(f"\n[Info] Device utilisé : {params['device']}")

ImageFile.LOAD_TRUNCATED_IMAGES = True

train_loader, val_loader, test_loader, classes = preparer_dataloaders(clean_path, params)
print(f"[Info] Classes : {classes}")

model = construire_modele(len(classes), params)
best_model_path, historique, best_val_acc = entrainer(model, train_loader, val_loader, params)

loader_test_final = test_loader if test_loader else val_loader
test_acc, rapport = tester(model, best_model_path, loader_test_final, classes, params["device"])

sauvegarder_dans_drive(df_stats, params, historique, test_acc, rapport, best_model_path)

In [ ]:
"""
==========================================================
PIPELINE ResNet50 - CLASSIFICATION (Google Colab, GPU)
==========================================================
Fine-tuning complet — sauvegarde directement dans
Drive/MonProjet/ResNet50(2)/
==========================================================
"""

!pip install -q torch torchvision scikit-learn

import os, json, shutil, random, hashlib, time
from pathlib import Path
from collections import defaultdict

from PIL import Image, ImageFile
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

# ==================================================================
# 1) MONTAGE DRIVE
# ==================================================================
from google.colab import drive
drive.mount('/content/drive')

# ==================================================================
# 2) CONFIGURATION
# ==================================================================
DATASET_PATH = "/content/drive/MyDrive/MonProjet/NewDataSet/concrete-damage-v3"
WORK_PATH    = "/content/newdataset_clean_resnet50"
IMG_SIZE     = 224   # résolution standard ResNet
SPLITS       = ["train", "val", "test"]

DRIVE_BASE = Path("/content/drive/MyDrive/MonProjet/ResNet50(2)")
for d in ["Dataset", "Hyperparametres", "Modeles", "Resultats"]:
    (DRIVE_BASE / d).mkdir(parents=True, exist_ok=True)

print(f"[Info] Dataset source : {DATASET_PATH}")
print(f"[Info] Existe ? {Path(DATASET_PATH).exists()}")
print(f"[Info] GPU disponible : {torch.cuda.is_available()}")

# ==================================================================
# 3) ANALYSE EXPLORATOIRE
# ==================================================================
def analyser_dataset(dataset_path):
    print("="*60); print("ANALYSE DU DATASET"); print("="*60)
    stats = []
    for split in SPLITS:
        split_path = Path(dataset_path) / split
        if not split_path.exists():
            print(f"[!] Dossier '{split}' introuvable, ignoré.")
            continue
        for cls_dir in [d for d in split_path.iterdir() if d.is_dir()]:
            images = list(cls_dir.glob("*.*"))
            widths, heights, sizes_ko = [], [], []
            for img_path in images[:200]:
                try:
                    with Image.open(img_path) as img:
                        w, h = img.size
                        widths.append(w); heights.append(h)
                    sizes_ko.append(os.path.getsize(img_path) / 1024)
                except Exception:
                    continue
            stats.append({
                "split": split, "classe": cls_dir.name, "n_images": len(images),
                "largeur_moy": sum(widths)/len(widths) if widths else None,
                "hauteur_moy": sum(heights)/len(heights) if heights else None,
                "poids_moy_ko": sum(sizes_ko)/len(sizes_ko) if sizes_ko else None,
            })
    df = pd.DataFrame(stats)
    print(df.to_string(index=False))
    if not df.empty:
        train_df = df[df["split"] == "train"]
        if not train_df.empty:
            plt.figure(figsize=(8, 4))
            plt.bar(train_df["classe"], train_df["n_images"])
            plt.xticks(rotation=45, ha="right")
            plt.title("Distribution des classes (train)")
            plt.ylabel("Nombre d'images")
            plt.tight_layout()
            plt.savefig("/content/distribution_classes_resnet50.png")
            plt.close()
            ratio = train_df["n_images"].max() / train_df["n_images"].min()
            print(f"\nRatio classe max/min : {ratio:.2f}")
    return df

# ==================================================================
# 4) DATA CLEANING
# ==================================================================
def detecter_images_corrompues(dataset_path):
    corrompues = []
    for split in SPLITS:
        split_path = Path(dataset_path) / split
        if not split_path.exists():
            continue
        for img_path in split_path.rglob("*.*"):
            try:
                with Image.open(img_path) as img:
                    img.load()
            except Exception:
                corrompues.append(img_path)
    print(f"Images corrompues/tronquées détectées : {len(corrompues)}")
    return corrompues

def detecter_doublons(dataset_path):
    hashes = defaultdict(list)
    for split in SPLITS:
        split_path = Path(dataset_path) / split
        if not split_path.exists():
            continue
        for img_path in split_path.rglob("*.*"):
            try:
                with open(img_path, "rb") as f:
                    h = hashlib.md5(f.read()).hexdigest()
                hashes[h].append(img_path)
            except Exception:
                continue
    doublons = []
    for h, paths in hashes.items():
        if len(paths) > 1:
            doublons.extend(paths[1:])
    print(f"Doublons détectés : {len(doublons)}")
    return doublons

def nettoyer_dataset(dataset_path, work_path):
    corrompues = set(detecter_images_corrompues(dataset_path))
    doublons = set(detecter_doublons(dataset_path))
    a_exclure = corrompues | doublons
    if os.path.exists(work_path):
        shutil.rmtree(work_path)
    n_copiees, n_exclues = 0, 0
    for split in SPLITS:
        split_path = Path(dataset_path) / split
        if not split_path.exists():
            continue
        for img_path in split_path.rglob("*.*"):
            if img_path in a_exclure:
                n_exclues += 1
                continue
            rel = img_path.relative_to(dataset_path)
            dest = Path(work_path) / rel
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(img_path, dest)
            n_copiees += 1
    print(f"\nNettoyage terminé : {n_copiees} images copiées, {n_exclues} exclues.")
    return work_path

def creer_val_si_absent(work_path, ratio_val=0.2, seed=42):
    random.seed(seed)
    train_path = Path(work_path) / "train"
    val_path = Path(work_path) / "val"
    if val_path.exists() and any(val_path.iterdir()):
        print("[Split] 'val' déjà présent.")
        return
    if not train_path.exists():
        return
    print(f"[Split] Création automatique de val ({ratio_val:.0%} de train).")
    for cls_dir in train_path.iterdir():
        if not cls_dir.is_dir():
            continue
        images = list(cls_dir.glob("*.*"))
        random.shuffle(images)
        n_val = max(1, int(len(images) * ratio_val))
        dest_dir = val_path / cls_dir.name
        dest_dir.mkdir(parents=True, exist_ok=True)
        for img in images[:n_val]:
            shutil.move(str(img), str(dest_dir / img.name))

# ==================================================================
# 5) MEILLEURS HYPERPARAMÈTRES (ResNet50, fine-tuning complet)
# ==================================================================
def obtenir_best_params():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    return dict(
        model="resnet50 (ImageNet pretrained)",
        imgsz=IMG_SIZE,
        epochs=30,
        batch=32,                  # ResNet50 (25M params) tient bien un batch de 32 sur T4
        optimizer="AdamW",
        lr0=0.0003,
        weight_decay=1e-4,
        scheduler="CosineAnnealingLR",
        label_smoothing=0.1,
        patience=7,
        augment=True,
        device=device,
        seed=42,
    )

# ==================================================================
# 6) DATALOADERS
# ==================================================================
def preparer_dataloaders(work_path, params):
    mean = [0.485, 0.456, 0.406]
    std  = [0.229, 0.224, 0.225]

    train_tf = transforms.Compose([
        transforms.RandomResizedCrop(params["imgsz"], scale=(0.8, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])
    eval_tf = transforms.Compose([
        transforms.Resize(int(params["imgsz"] * 1.14)),
        transforms.CenterCrop(params["imgsz"]),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])

    train_ds = datasets.ImageFolder(str(Path(work_path) / "train"), transform=train_tf)
    val_ds   = datasets.ImageFolder(str(Path(work_path) / "val"), transform=eval_tf)

    test_path = Path(work_path) / "test"
    test_ds = datasets.ImageFolder(str(test_path), transform=eval_tf) if test_path.exists() else None

    train_loader = DataLoader(train_ds, batch_size=params["batch"], shuffle=True, num_workers=2)
    val_loader   = DataLoader(val_ds, batch_size=params["batch"], shuffle=False, num_workers=2)
    test_loader  = DataLoader(test_ds, batch_size=params["batch"], shuffle=False, num_workers=2) if test_ds else None

    return train_loader, val_loader, test_loader, train_ds.classes

# ==================================================================
# 7) MODELE
# ==================================================================
def construire_modele(num_classes, params):
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)
    return model.to(params["device"])

# ==================================================================
# 8) ENTRAINEMENT AVEC EARLY STOPPING
# ==================================================================
def entrainer(model, train_loader, val_loader, params):
    device = params["device"]
    optimizer = torch.optim.AdamW(model.parameters(), lr=params["lr0"], weight_decay=params["weight_decay"])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=params["epochs"])
    criterion = nn.CrossEntropyLoss(label_smoothing=params["label_smoothing"])

    best_val_acc = 0.0
    epochs_sans_amelioration = 0
    historique = {"epoch": [], "train_loss": [], "val_loss": [], "val_acc": []}
    best_model_path = "/content/best_resnet50.pt"

    for epoch in range(1, params["epochs"] + 1):
        t0 = time.time()
        model.train()
        train_loss = 0.0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * imgs.size(0)
        train_loss /= len(train_loader.dataset)

        model.eval()
        val_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                outputs = model(imgs)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * imgs.size(0)
                preds = outputs.argmax(dim=1)
                correct += (preds == labels).sum().item()
                total += labels.size(0)
        val_loss /= len(val_loader.dataset)
        val_acc = correct / total

        scheduler.step()
        dt = time.time() - t0
        print(f"Epoch {epoch}/{params['epochs']} - train_loss={train_loss:.4f} "
              f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} ({dt:.1f}s)")

        historique["epoch"].append(epoch)
        historique["train_loss"].append(train_loss)
        historique["val_loss"].append(val_loss)
        historique["val_acc"].append(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            epochs_sans_amelioration = 0
            torch.save(model.state_dict(), best_model_path)
        else:
            epochs_sans_amelioration += 1
            if epochs_sans_amelioration >= params["patience"]:
                print(f"\n[EarlyStopping] Arrêt à l'epoch {epoch} (meilleur val_acc={best_val_acc:.4f})")
                break

    return best_model_path, historique, best_val_acc

# ==================================================================
# 9) TEST FINAL
# ==================================================================
def tester(model, best_model_path, loader, classes, device):
    model.load_state_dict(torch.load(best_model_path))
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            outputs = model(imgs)
            preds = outputs.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())

    acc = accuracy_score(all_labels, all_preds)
    print(f"\nTest accuracy : {acc:.4f}")

    labels_indices = list(range(len(classes)))
    print(classification_report(all_labels, all_preds, labels=labels_indices,
                                 target_names=classes, zero_division=0))

    cm = confusion_matrix(all_labels, all_preds, labels=labels_indices)
    plt.figure(figsize=(6, 5))
    plt.imshow(cm, cmap="Blues")
    plt.title("Matrice de confusion")
    plt.colorbar()
    plt.xticks(range(len(classes)), classes, rotation=45, ha="right")
    plt.yticks(range(len(classes)), classes)
    for i in range(len(classes)):
        for j in range(len(classes)):
            plt.text(j, i, cm[i, j], ha="center", va="center")
    plt.xlabel("Prédiction"); plt.ylabel("Réel"); plt.tight_layout()
    plt.savefig("/content/confusion_matrix_resnet50.png")
    plt.close()

    rapport = classification_report(all_labels, all_preds, labels=labels_indices,
                                     target_names=classes, output_dict=True, zero_division=0)
    return acc, rapport

# ==================================================================
# 10) SAUVEGARDE DANS DRIVE
# ==================================================================
def sauvegarder_dans_drive(df_stats, params, historique, test_acc, rapport, best_model_path):
    if df_stats is not None and not df_stats.empty:
        df_stats.to_csv(DRIVE_BASE / "Dataset" / "analyse_dataset.csv", index=False)
    dist_png = Path("/content/distribution_classes_resnet50.png")
    if dist_png.exists():
        shutil.copy2(dist_png, DRIVE_BASE / "Dataset" / "distribution_classes.png")

    params_propre = {k: str(v) for k, v in params.items()}
    with open(DRIVE_BASE / "Hyperparametres" / "hyperparametres_resnet50.json", "w") as f:
        json.dump(params_propre, f, indent=2, ensure_ascii=False)

    shutil.copy2(best_model_path, DRIVE_BASE / "Modeles" / "best_resnet50.pt")

    df_hist = pd.DataFrame(historique)
    df_hist.to_csv(DRIVE_BASE / "Resultats" / "results.csv", index=False)

    plt.figure(figsize=(8, 4))
    plt.plot(df_hist["epoch"], df_hist["train_loss"], label="train_loss")
    plt.plot(df_hist["epoch"], df_hist["val_loss"], label="val_loss")
    plt.legend(); plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title("Courbes de loss")
    plt.tight_layout()
    plt.savefig(DRIVE_BASE / "Resultats" / "results.png")
    plt.close()

    cm_png = Path("/content/confusion_matrix_resnet50.png")
    if cm_png.exists():
        shutil.copy2(cm_png, DRIVE_BASE / "Resultats" / "confusion_matrix.png")

    with open(DRIVE_BASE / "Resultats" / "classification_report.json", "w") as f:
        json.dump(rapport, f, indent=2, ensure_ascii=False)

    readme = f"""# ResNet50(2) — Classification

## Resultats
- Test accuracy : {test_acc:.4f}
- Voir Resultats/classification_report.json pour le détail par classe
"""
    with open(DRIVE_BASE / "README_ResNet50.md", "w") as f:
        f.write(readme)

    print(f"\n✅ Tout est dans ton Drive : {DRIVE_BASE}")


# ==================================================================
# 11) EXECUTION COMPLETE
# ==================================================================
df_stats = analyser_dataset(DATASET_PATH)
clean_path = nettoyer_dataset(DATASET_PATH, WORK_PATH)
creer_val_si_absent(clean_path, ratio_val=0.2)

params = obtenir_best_params()
print(f"\n[Info] Device utilisé : {params['device']}")

ImageFile.LOAD_TRUNCATED_IMAGES = True

train_loader, val_loader, test_loader, classes = preparer_dataloaders(clean_path, params)
print(f"[Info] Classes : {classes}")

model = construire_modele(len(classes), params)
best_model_path, historique, best_val_acc = entrainer(model, train_loader, val_loader, params)

loader_test_final = test_loader if test_loader else val_loader
test_acc, rapport = tester(model, best_model_path, loader_test_final, classes, params["device"])

sauvegarder_dans_drive(df_stats, params, historique, test_acc, rapport, best_model_path)

In [6]:
!find / -name "*.pth" 2>/dev/null

/usr/lib/python3/dist-packages/distutils-precedence.pth
/usr/local/lib/python3.13/dist-packages/distutils-precedence.pth
/usr/local/lib/python3.13/dist-packages/_zz_cuml_accel.pth
/usr/local/lib/python3.13/dist-packages/_rapids_dask_dependency.pth
/usr/local/lib/python3.13/dist-packages/_numba_cuda_redirector.pth


In [7]:
!find / -name "*.ipynb" 2>/dev/null

/root/.julia/packages/Interpolations/rXcvn/doc/Plotting examples.ipynb
/root/.julia/packages/Interpolations/rXcvn/doc/Interpolations.jl.ipynb
/usr/local/lib/python3.13/dist-packages/panel/tests/ui/io/app.ipynb
/usr/local/lib/python3.13/dist-packages/notebook/bundler/tests/resources/empty.ipynb
/usr/local/lib/python3.13/dist-packages/nbclassic/bundler/tests/resources/empty.ipynb
/usr/local/lib/python3.13/dist-packages/cuml/experimental/hyperparams/HPO_demo.ipynb
/usr/local/lib/python3.13/dist-packages/geemap/data/template/template.ipynb
/usr/local/lib/python3.13/dist-packages/holoviews/examples/getting_started/5-Live_Data.ipynb
/usr/local/lib/python3.13/dist-packages/holoviews/examples/getting_started/1-Introduction.ipynb
/usr/local/lib/python3.13/dist-packages/holoviews/examples/getting_started/2-Customization.ipynb
/usr/local/lib/python3.13/dist-packages/holoviews/examples/getting_started/4-Gridded_Datasets.ipynb
/usr/local/lib/python3.13/dist-packages/holoviews/examples/getting_start

In [9]:
from google.colab import drive
drive.mount('/content/drive')
!find /content/drive -name "*.ipynb" -not -path "*/.ipynb_checkpoints/*"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/Colab Notebooks/Untitled0.ipynb
/content/drive/MyDrive/Colab Notebooks/Untitled2.ipynb
/content/drive/MyDrive/Colab Notebooks/Untitled3.ipynb
/content/drive/MyDrive/Colab Notebooks/S1_Dhia_KMeans_Colab_leger.ipynb
/content/drive/MyDrive/Colab Notebooks/Untitled4.ipynb
/content/drive/MyDrive/Colab Notebooks/Untitled5.ipynb
/content/drive/MyDrive/Colab Notebooks/Untitled1.ipynb


In [10]:
import json, glob

for f in sorted(glob.glob("/content/drive/MyDrive/Colab Notebooks/*.ipynb")):
    nb = json.load(open(f))
    code = " ".join("".join(c["source"]) for c in nb["cells"] if c["cell_type"] == "code")
    mots = [m for m in ["torch", "tensorflow", "keras", "sklearn", "CNN", "LSTM", "Conv2d", "KMeans"] if m in code]
    print(f.split("/")[-1], "->", len(nb["cells"]), "cellules |", mots)


S1_Dhia_KMeans_Colab_leger.ipynb -> 26 cellules | ['sklearn', 'KMeans']
Untitled0.ipynb -> 25 cellules | ['tensorflow', 'keras', 'sklearn']
Untitled1.ipynb -> 19 cellules | ['torch', 'sklearn']
Untitled2.ipynb -> 1 cellules | []
Untitled3.ipynb -> 1 cellules | []
Untitled4.ipynb -> 2 cellules | []
Untitled5.ipynb -> 1 cellules | []


In [8]:
from google.colab import drive
drive.mount('/content/drive')
!find /content/drive -name "*.pth" -o -name "*.py" -o -name "*.pt"

Mounted at /content/drive
/content/drive/MyDrive/MonProjet/Yolo26/Modeles/best.pt
/content/drive/MyDrive/MonProjet/YOLO26(2)/Modeles/best (1).pt
/content/drive/MyDrive/MonProjet/YOLO26(2)/Modeles/best_YOLO26_2.pt
/content/drive/MyDrive/MonProjet/YOLO26(2)/Modeles/last_YOLO26_2.pt
/content/drive/MyDrive/MonProjet/EfficientNetB0(2)/Modeles/best_efficientnetb0.pt
/content/drive/MyDrive/MonProjet/EfficientNetB2(2)/Modeles/best_efficientnetb2.pt
/content/drive/MyDrive/MonProjet/ResNet50(2)/Modeles/best_resnet50.pt
